## Canadian Student Tobacco, Alcohol and Drugs Survey (CSTADS) 2021-2022 Public Use Microdata File (PUMF)

In [ ]:
!pip install imblearn

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler
import joblib
import pandas as pd

data = pd.read_csv('Data.csv')
data = data.fillna(data.mode().iloc[0])
data_encoded = data.replace({'Yes': 1, 'No': 0}).infer_objects(copy=False)

# selected_vars = [
#     # smoking
#     'SS_010', # have they ever tried smoking
#     'SS_020', # how old when first tried smoking
#     'SS_030', # have they ever tried smoking a whole cigarette
#     'SS_040', # have they smoked 100 or more whole cigarettes in their life
#     'WP_040a', # # of whole cigarette smoked on Sunday (over the last 7 days)
#     'WP_040b', # # of whole cigarette smoked on Monday (over the last 7 days)
#     'WP_040c', # # of whole cigarette smoked on Tuesday (over the last 7 days)
#     'WP_040d', # # of whole cigarette smoked on Wednesday (over the last 7 days)
#     'WP_040e', # # of whole cigarette smoked on Thursday (over the last 7 days)
#     'WP_040f', # # of whole cigarette smoked on Friday (over the last 7 days)
#     'WP_040g', # # of whole cigarette smoked on Saturday (over the last 7 days)

#     # alcohol
#     'ALC_010', # have they ever tried alcohol (more than a sip)
#     'ALC_020', # have they ever tried alcohol (more than a sip) in the last year
#     'ALC_030', # age at first drink of alcohol (more than a sip)
#     'ALC_040', # how often did they have a drink of alchol (more than a sip) in the past 30 days
#     'ALC_050', # how often did they have 5 or more drinks of alcohol on one occasion (in the last year)
#     'ALC_060', # age when they first had 5 or more drinks of alcohol in one occasion
#     'ALC_071', # how often did they have 5 or more drinks of alcohol on one occasion (in the last 30 days)

#     # cannabis
#     'CAN_010',
#     'CAN_020',
#     'CAN_030',
#     'CAN_040',
#     'CAN_050',
#     'CAN_060',
#     'CAN_070',
#     'CAN_080',
#     'CAN_091',
#     'CAN_092',
#     'CAN_100',
#     'CAN_110',
#     'CAN_121',
#     'CAN_130',
#     'CAN_140',

#     # other
#     'BEH_010', # behaviour indicators
#     'GRADE', # grade level
#     'DVURBAN', # where they live; urban/rural
#     'ELC_026a', # e cigarettes
#     'PH_010', # physical activity levels,
#     'DVDESCRIBE', # race,
#     'DVORIENT' # sexual orientation
# ]

# testing below
selected_vars = [
    'GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT', # Demographics
    'BEH_010', 'PH_010',                         # Behavioral
    'ALC_010', 'ALC_040',                        # Alcohol
    'CAN_010',                                   # Cannabis
    'SS_010',                                    # Smoking
    'ELC_026a'                                   # E-cigarettes
]

# data_encoded['target_column'] = (
#     data_encoded['GH_010'] & # general physical health
#     data_encoded['GH_020'] # general mental health
# ).astype(int)


# testing below
data_encoded['target_column'] = (
    data_encoded['ALC_020'] | data_encoded['CAN_010'] | data_encoded['SS_040']
).astype(int)


X = data_encoded[selected_vars]
y = data_encoded['target_column'] # substance use

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

selector = SelectKBest(f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)

selected_features_indices = selector.get_support(indices=True)
selected_features = [selected_vars[i] for i in selected_features_indices]
print("Selected Features:", selected_features)

X_test_selected = selector.transform(X_test)

# smote = SMOTE(random_state=42)
smote_tomek = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train_selected, y_train)

param_grid = {
    'C': [0.1, 1, 10, 100],
    # 'solver': ['liblinear', 'lbfgs']
    'solver': ['liblinear']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced'), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_resampled, y_train_resampled)

best_model = grid.best_estimator_
print("Best Model:", best_model)

# best_model.fit(X_train_resampled, y_train_resampled)

y_pred = best_model.predict(X_test_selected)
y_pred_prob = best_model.predict_proba(X_test_selected)[:, 1] * 100

# model = LogisticRegression(max_iter=1000)
# model.fit(X_train_resampled, y_train_resampled)

# y_pred = model.predict(X_test_selected)
# y_pred_prob = model.predict_proba(X_test_selected)[:, 1] * 100

print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification report:\n', classification_report(y_test, y_pred))

output_df = pd.DataFrame(X_test_selected, columns=selected_features)
output_df['Predicted Probability (%)'] = y_pred_prob
print(output_df[['Predicted Probability (%)']].head())

joblib.dump(best_model, 'logistic_model.pkl')

Selected Features: ['GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT', 'PH_010', 'ALC_010', 'ALC_040', 'CAN_010', 'SS_010', 'ELC_026a']


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Best Model: LogisticRegression(C=100, class_weight='balanced', max_iter=1000,
                   solver='liblinear')
Accuracy: 0.605106661574554
Confusion matrix:
 [[   0    0    6    3    0    5    2    0    0    0    0    0    0    0
     0    0    0]
 [  16    0   26   77    9   71   42    0    5    4    0   20    0    2
     2    1    0]
 [  10    0   26   14    5   56    8    0    1    0    0    5    0    0
     0    0    0]
 [   0    0    0   20    0    0    3    0    0    0    0    0    0    3
     0    0    0]
 [  29    0   99  115   72  304   36    0    6    1    0   39    0    6
     2    1    6]
 [   3    0    5    6    2   38    2    0    3    0    0    0    0    0
     1    0    0]
 [   0    0    0    5    0    0    4    0    0    1    0    0    0    0
     0    0    0]
 [   6    0    8   25    6   36    9    0    1    0    0    2    0    2
     2    0    2]
 [   1    0    6   13    2    3    8    9  260    0    0   54    0    0
    15   16    0]
 [   0    0    0    0    0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['logistic_model.pkl']

In [ ]:
print(data_encoded['target_column'].value_counts())

### Random Forest

In [ ]:
!nvidia-smi
!pip install cudf-cu11 cuml-cu11 --extra-index-url=https://pypi.ngc.nvidia.com

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import joblib

# data = pd.read_csv('Data.csv')
# data = data.fillna(data.mode().iloc[0])
# data_encoded = data.replace({'Yes': 1, 'No': 0}).infer_objects(copy=False)

def replaceTarget(value):
    return 1 if 1 <= value <= 3 else 0

def clean_data(data):
    filtered_data = data
    conditions = (
        (~data['GH_010'].isin([99, 6])) &
        (~data['GH_020'].isin([99, 6]))
    )
    filtered_data = data[conditions]
    filtered_data['GH_010'] = filtered_data['GH_010'].apply(replaceTarget)
    filtered_data['GH_020'] = filtered_data['GH_020'].apply(replaceTarget)
    return filtered_data


data = pd.read_csv('Data.csv')
data = data.dropna()
filtered_data = clean_data(data)

filtered_data['target_column'] = (
        filtered_data['GH_020'] & filtered_data['GH_010']
)

selected_vars = [
    'PROVID', 'GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT',  # Demographics
    'BEH_010', 'PH_010',  # Behavioral
    'ALC_040', 'ALC_010',  # Alcohol
    'CAN_010',  # Cannabis
    'SS_010', 'SS_030', # Smoking
    'ELC_026a', 'ELC_026c'  # Other features
]
X = filtered_data[selected_vars]
y = filtered_data['target_column']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

selector = SelectKBest(f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
selected_features_indices = selector.get_support(indices=True)
selected_features = [selected_vars[i] for i in selected_features_indices]
print("Selected Features:", selected_features)

X_test_selected = selector.transform(X_test)

smote_tomek = SMOTETomek(smote=SMOTE(k_neighbors=1), random_state=42)
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train_selected, y_train)

param_grid = {
    'n_estimators': [50, 100, 200],
    # 'n_estimators': [50, 100],
    'max_depth': [None, 10, 20, 30],
    # 'max_depth': [None, 10],
    'min_samples_split': [2, 5, 10],
    # 'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    # 'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
    # 'bootstrap': [True]
}

# grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=param_grid, n_iter=100, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
grid.fit(X_train_resampled, y_train_resampled)

# best_rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=1, min_samples_split=2, bootstrap=True)
# best_rf_model.fit(X_train_resampled, y_train_resampled)
best_rf_model = grid.best_estimator_
print("Best Model Parameters:", grid.best_params_)

y_pred = best_rf_model.predict(X_test_selected)
y_pred_prob = best_rf_model.predict_proba(X_test_selected)[:, 1] * 100

print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification report:\n', classification_report(y_test, y_pred))

feature_importance = pd.Series(best_rf_model.feature_importances_, index=selected_features).sort_values(ascending=False)
print("Feature Importance:\n", feature_importance)

joblib.dump(best_rf_model, 'random_forest_model.pkl')

output_df = pd.DataFrame(X_test_selected, columns=selected_features)
output_df['Predicted Probability (%)'] = y_pred_prob
print(output_df[['Predicted Probability (%)']].head())

#### Random Forest with CUDA

#### XGBoost


In [ ]:
!pip install scikit-learn==1.3.1

!pip install xgboost==2.1.3

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, recall_score, precision_score
from xgboost import DMatrix, cv
import random

def replaceTarget(value): return 1 if 1 <= value <= 3 else 0

def clean_data(data):
    filtered_data = data
    conditions = (
        (~data['GH_010'].isin([99, 6])) &
        (~data['GH_020'].isin([99, 6]))
    )
    filtered_data = data[conditions].copy()
    filtered_data.loc[:, 'GH_010'] = filtered_data['GH_010'].apply(replaceTarget)
    filtered_data.loc[:, 'GH_020'] = filtered_data['GH_020'].apply(replaceTarget)
    return filtered_data

data = pd.read_csv('Data.csv')
filtered_data = clean_data(data)

filtered_data.loc[:, 'target_column'] = (
    filtered_data['GH_020'] | filtered_data['GH_010']
)

selected_vars = [
    'PROVID', 'GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT',  # Demographics
    'BEH_010', 'PH_010',  # Behavioral
    'ALC_040', 'ALC_010',  # Alcohol
    'CAN_010',  # Cannabis
    'SS_010', 'SS_030', # Smoking
    'ELC_026a', 'ELC_026c'  # Other features
]

X = filtered_data[selected_vars]
y = filtered_data['target_column']

# MARK:- running gridsearch below
'''
param_grid = {
    'n_estimators': [50, 10, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

gridsearch = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='accuracy')
gridsearch.fit(X_train, y_train)

best_model = gridsearch.best_estimator_
print('Best parameters: ', gridsearch.best_params_)

'''

# MARK:- using best params for model

def run_model():
    # test_size = round(random.uniform(0.05, 0.5), 2)
    test_size=0.30
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

    model = xgb.XGBClassifier()
    model.fit(X_train, y_train)

    best_model = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=1.0)
    best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    importance = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({'Feature': selected_vars, 'Importance': importance})
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    return mse, feature_importance_df, accuracy, f1, best_model, test_size, recall, precision

best_run = {
    'mse': float('inf'),
    'feature_importance_df': None,
    'accuracy': 0,
    'f1': 0,
    'best_model': None,
    'test_size': 0,
    'recall': 0,
    'precision': 0
}

for i in range(1):
    mse, feature_importance_df, accuracy, f1, best_model, test_size, recall, precision = run_model()

    if accuracy > best_run['accuracy']:
        best_run['accuracy'] = accuracy
        best_run['mse'] = mse
        best_run['feature_importance_df'] = feature_importance_df
        best_run['f1'] = f1
        best_run['best_model'] = best_model
        best_run['recall'] = recall
        best_run['precision'] = precision
        print(f'[!] New best model found with accuracy: {accuracy * 100:.2f}%, test size: {test_size}')

    print(f'{i + 1}. Running model... {accuracy * 100:.2f}%, test size: {test_size}')

print(f'F1 Score: {best_run["f1"]:.4f}')
print(f'Mean Squared Error: {best_run["mse"]:.4f}')
print(f'Accuracy: {best_run["accuracy"] * 100:.2f}%')
print('Feature importance:')
print(best_run['feature_importance_df'].sort_values(by='Importance', ascending=False), end='\n\n')

print(best_run)

# Neural Network



In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

def replaceTarget(value):
    return 1 if 1 <= value <= 3 else 0

def clean_data(data):
    # clean target columns
    filtered_data = data
    conditions = (
        (~data['GH_010'].isin([99, 6])) &
        (~data['GH_020'].isin([99, 6]))
    )
    filtered_data = data[conditions]
    filtered_data['GH_010'] = filtered_data['GH_010'].apply(replaceTarget)
    filtered_data['GH_020'] = filtered_data['GH_020'].apply(replaceTarget)

    return filtered_data


data = pd.read_csv('Data.csv')
filtered_data = clean_data(data)

filtered_data['target_column'] = (
        filtered_data['GH_020'] | filtered_data['GH_010']
)

selected_vars = [
    'PROVID', 'GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT',  # Demographics
    'BEH_010', 'PH_010',  # Behavioral
    'ALC_040', 'ALC_010',  # Alcohol
    'CAN_010',  # Cannabis
    'SS_010', 'SS_030', # Smoking
    'ELC_026a', 'ELC_026c'  # Other features
]
X = filtered_data[selected_vars]
y = filtered_data['target_column']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit and transform on training data
X_test = scaler.transform(X_test)       # Transform test data (using training parameters)

model = Sequential()
model.add(Dense(64, input_dim=len(selected_vars), activation='relu'))
model.add(Dropout(0.5))  # 30% dropout and play around with this value
model.add(Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)))
model.add(BatchNormalization())
model.add(Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid', kernel_regularizer=tf.keras.regularizers.l2(0.01)))



model.compile(optimizer='Adamax', loss='binary_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=10)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_split=0.2, callbacks=[early_stopping, reduce_lr])

# Evaluate the Model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

# Plot Training History
plt.figure(figsize=(12, 4))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()



In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

def replaceTarget(value):
    return 1 if 1 <= value <= 3 else 0

def clean_data(data):
    # clean target columns
    filtered_data = data
    conditions = (
        (~data['GH_010'].isin([99, 6])) &
        (~data['GH_020'].isin([99, 6]))
    )
    filtered_data = data[conditions]
    filtered_data['GH_010'] = filtered_data['GH_010'].apply(replaceTarget)
    filtered_data['GH_020'] = filtered_data['GH_020'].apply(replaceTarget)

    # clean other columns

    return filtered_data


data = pd.read_csv('Data.csv')
filtered_data = clean_data(data)

filtered_data['target_column'] = (
        filtered_data['GH_020'] & filtered_data['GH_010']
)

selected_vars = [
    'PROVID', 'GRADE', 'DVURBAN', 'DVDESCRIBE', 'DVORIENT',  # Demographics
    'BEH_010', 'PH_010',  # Behavioral
    'ALC_040', 'ALC_010',  # Alcohol
    'CAN_010',  # Cannabis
    'SS_010', 'SS_030', # Smoking
    'ELC_026a' # Other features
]
X = filtered_data[selected_vars]
y = filtered_data['target_column']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = Sequential()
model.add(Dense(64, input_dim=len(selected_vars), activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))



model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_split=0.3)

# Evaluate the Model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

# Plot Training History
plt.figure(figsize=(12, 4))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()



In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

def replaceTarget(value):
    return 1 if 1 <= value <= 3 else 0

def clean_data(data):
    # clean target columns
    filtered_data = data
    conditions = (
        (~data['GH_010'].isin([99, 6])) &
        (~data['GH_020'].isin([99, 6]))
    )
    filtered_data = data[conditions]
    filtered_data['GH_010'] = filtered_data['GH_010'].apply(replaceTarget)
    filtered_data['GH_020'] = filtered_data['GH_020'].apply(replaceTarget)


    # print(filtered_data['PH_010'].mode())
    filtered_data['PH_010'] = filtered_data['PH_010'].replace(99, 5)
    # print(filtered_data['ALC_040'].mode())
    filtered_data['ALC_040'] = filtered_data['ALC_040'].replace([99, 7], 96)
    # print(filtered_data['CAN_010'].mode())
    # print(filtered_data['SS_030'].mode())
    filtered_data['SS_030'] = filtered_data['SS_030'].replace(99, 96)
    # print(filtered_data['ELC_026a'].mode())
    # print(filtered_data['ELC_026c'].mode())

    # clean other columns

    return filtered_data





data = pd.read_csv('Data.csv')
filtered_data = clean_data(data)

filtered_data['target_column'] = (
        filtered_data['GH_020'] | filtered_data['GH_010']
)

selected_vars = [
    'PROVID', 'GRADE', 'DVURBAN', 'DVORIENT',  # Demographics
    'PH_010',  # Behavioral
    'ALC_040',  # Alcohol
    'CAN_010',  # Cannabis
    'SS_030',   # Smoking
    'ELC_026a', 'ELC_026c'  # Other features
]
X = filtered_data[selected_vars]
y = filtered_data['target_column']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


model = Sequential()
model.add(Dense(64, input_dim=len(selected_vars), activation='relu'))
model.add(BatchNormalization())
model.add(Dense(1024, activation='relu'))
model.add(Dense(512, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))



model.compile(optimizer='RMSprop', loss='binary_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=10)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_split=0.25, callbacks=[early_stopping, reduce_lr])

# Evaluate the Model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

# Plot Training History
plt.figure(figsize=(12, 4))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

